# 06 — Model Evaluation

**Research**: Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF

**Objective**: Evaluate the best tuned models on the isolated test dataset.

**Models**:
1. Multinomial Naive Bayes
2. Logistic Regression
3. Linear SVM

**Evaluation Metrics**:
- Accuracy, Precision, Recall, F1 Score (Macro & Per-Class)
- Confusion Matrix Heatmaps
- ROC Curves (One-vs-Rest)

---

## 1. Setup

In [1]:
import sys
import os
import joblib
import warnings

# Suppress specific scikit-learn warnings if desired
warnings.filterwarnings('ignore', category=UserWarning)

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import project modules
from src.config.settings import (
    EVALUATION_REPORTS_DIR,
    EVAL_COMPARISON_DIR,
    NB_BEST_MODEL_PATH,
    LR_BEST_MODEL_PATH,
    SVM_BEST_MODEL_PATH,
)
from src.config.evaluation_config import EvaluationConfig
from src.utils.load_features import load_features
from src.evaluation.evaluator import evaluate_model
from src.evaluation.model_comparison import generate_comparison_dataframe, generate_all_comparisons
from src.evaluation.report_generator import generate_evaluation_summary
from src.evaluation.persistence import save_evaluation_artifacts

print('Setup complete.')

Setup complete.


---
## 2. Load Models & Test Dataset

In [2]:
# Load test features and labels
# (X_train and y_train are intentionally ignored here to prevent data leakage)
_, X_test, _, y_test = load_features()

print(f'\nTesting set: X={X_test.shape}, y={len(y_test):,}')

# Load tuned models
print('\nLoading tuned models...')
nb_model = joblib.load(NB_BEST_MODEL_PATH)
lr_model = joblib.load(LR_BEST_MODEL_PATH)
svm_model = joblib.load(SVM_BEST_MODEL_PATH)
print('✓ All models loaded.')

✓ Artifacts loaded: X_train=(33244, 22695), X_test=(8312, 22695), vocab=22,695
✓ Features validated: X_train=(33244, 22695), X_test=(8312, 22695), labels=6 classes

Testing set: X=(8312, 22695), y=8,312

Loading tuned models...
✓ All models loaded.


---
## 3. Configure Evaluation

In [3]:
config = EvaluationConfig(
    average="macro",
    save_figures=True,
    save_reports=True,
)

print('Evaluation Configuration:')
for k, v in config.to_dict().items():
    print(f'  {k}: {v}')

Evaluation Configuration:
  average: macro
  output_directory: /home/zapp/Kampus/PM/reports/evaluation
  save_figures: True
  save_reports: True
  figure_format: png
  dpi: 300
  class_labels: ['normal', 'hate_speech', 'insult', 'harassment', 'threat', 'sexually_explicit']


---
## 4. Evaluate Models

In [4]:
results = []

# 1. Evaluate Naive Bayes
nb_results = evaluate_model(nb_model, 'Naive Bayes', X_test, y_test, config)
results.append(nb_results)
print('-' * 40)

# 2. Evaluate Logistic Regression
lr_results = evaluate_model(lr_model, 'Logistic Regression', X_test, y_test, config)
results.append(lr_results)
print('-' * 40)

# 3. Evaluate SVM
svm_results = evaluate_model(svm_model, 'Support Vector Machine', X_test, y_test, config)
results.append(svm_results)

Evaluating Naive Bayes...
  Generating predictions...
  Calculating metrics...
  Calculating ROC curves (OvR)...
  ✓ Evaluation complete (0.30s)
    F1 (Macro): 0.4279
    Accuracy:   0.7897
----------------------------------------
Evaluating Logistic Regression...
  Generating predictions...
  Calculating metrics...
  Calculating ROC curves (OvR)...
  ✓ Evaluation complete (0.24s)
    F1 (Macro): 0.4837
    Accuracy:   0.7887
----------------------------------------
Evaluating Support Vector Machine...
  Generating predictions...
  Calculating metrics...
  Calculating ROC curves (OvR)...
  ✓ Evaluation complete (22.44s)
    F1 (Macro): 0.5042
    Accuracy:   0.8103


---
## 5. Model Comparison

In [5]:
print('\nAggregating comparison metrics...')
comparison_df = generate_comparison_dataframe(results)
display(comparison_df[['f1_macro', 'accuracy', 'precision_macro', 'recall_macro']])

print('\nGenerating comparison plots...')
generate_all_comparisons(comparison_df, EVAL_COMPARISON_DIR)
print('✓ Comparison plots generated.')


Aggregating comparison metrics...


,f1_macro,accuracy,precision_macro,recall_macro
Model,,,,
Support Vector Machine,0.504206,0.810274,0.752181,0.447222
Logistic Regression,0.483658,0.788739,0.624221,0.437391
Naive Bayes,0.427897,0.789702,0.612622,0.390788



Generating comparison plots...
  ✓ Comparison plot saved: /home/zapp/Kampus/PM/reports/evaluation/comparison/comparison_f1_macro.png
  ✓ Comparison plot saved: /home/zapp/Kampus/PM/reports/evaluation/comparison/comparison_accuracy.png
  ✓ Comparison plot saved: /home/zapp/Kampus/PM/reports/evaluation/comparison/comparison_precision_macro.png
  ✓ Comparison plot saved: /home/zapp/Kampus/PM/reports/evaluation/comparison/comparison_recall_macro.png
✓ Comparison plots generated.


---
## 6. Save Reports & Artifacts

In [6]:
print('Generating and saving comprehensive reports...\n')

summary_md = generate_evaluation_summary(comparison_df, results, len(y_test))

save_evaluation_artifacts(
    results=results,
    comparison_df=comparison_df,
    summary_md=summary_md,
    config=config
)

print(f'\n✓ All evaluation artifacts saved to: {EVALUATION_REPORTS_DIR}')

Generating and saving comprehensive reports...

  ✓ Saved summary and comparison CSV to /home/zapp/Kampus/PM/reports/evaluation
  ✓ Confusion matrix plot saved: /home/zapp/Kampus/PM/reports/evaluation/confusion_matrices/naive_bayes_cm.png
  ✓ ROC curve plot saved: /home/zapp/Kampus/PM/reports/evaluation/roc_curves/naive_bayes_roc.png
  ✓ Confusion matrix plot saved: /home/zapp/Kampus/PM/reports/evaluation/confusion_matrices/logistic_regression_cm.png
  ✓ ROC curve plot saved: /home/zapp/Kampus/PM/reports/evaluation/roc_curves/logistic_regression_roc.png
  ✓ Confusion matrix plot saved: /home/zapp/Kampus/PM/reports/evaluation/confusion_matrices/support_vector_machine_cm.png
  ✓ ROC curve plot saved: /home/zapp/Kampus/PM/reports/evaluation/roc_curves/support_vector_machine_roc.png
  ✓ All individual model artifacts saved.

✓ All evaluation artifacts saved to: /home/zapp/Kampus/PM/reports/evaluation


---
## Summary

Model Evaluation is complete. All outputs can be found in `reports/evaluation/`.

**Generated Artifacts**:
- `evaluation_summary.md`: Main Markdown summary
- `model_comparison.csv`: CSV of compared metrics
- `comparison/`: Bar charts comparing algorithms
- `confusion_matrices/`: CM CSVs and heatmaps (PNG)
- `roc_curves/`: OvR ROC curve plots (PNG)
- `classification_reports/`: Per-model metrics (CSV/MD)

**Next Step**: `07_error_analysis.ipynb` — Error Analysis